# Extra: selection of TF-IDF components based on entropy

This notebook is a small extension of the original project. Here we want to further analyze the network built using tf-idf embeddings of abstracts. In particular, since tf-idf embedding vectors have a huge number of components, we want to see if filtering some of these components can lead to a reduction of the noise present in the embedding vectors, and thus bring to an improvement of classification performances.

The selection of components is based on entropy: for each component of the vector embeddings, the information entropy is calculated across all papers is calculated using:

\begin{equation}
    H = -\sum_{i=1}^N p_ilog(p_i)
\end{equation}

where $N = 25877$ is the total number of papers, while $p_i$ represents the value of a certain component of embedding vectors in paper $i$ when the values of that component are normalized across all papers so that $\sum_{i=1}^N p_i = 1$.

After calculating the value of entropy for all embedding components, only the components with entropy higher than a certain threshold are kept. Finally, the distance matrix, the adjacency matrix and the corresponding network are built from tf-idf embeddings filtered in this way, and all the algorithms already employed in the original project are tried to divide the network into communities, to see if we have an improvement of classification after the selection of components.

## Entropy calculation

First of all we want to compute the entropy of all components of embedding vectors, and save in a npz file the array containing all values of entropy.

In [ ]:
from components_selection import entropy_component
from scipy.sparse import load_npz
import numpy as np

tf_idf_embeddings = load_npz("../embeddings/abstract_embeddings_tfidf.npz")

In [3]:
for i in range(223):
    #select 250 columns, since 223 * 250 = 55750, number of columns of the original matrix
    batch_of_columns = tf_idf_embeddings[:, (i * 250):((i+1) * 250)].toarray()

    entropy = entropy_component(batch_of_columns)

    #append the arrays obtained in the different cycles
    if i == 0:
        full_entropy_array = entropy
    else:
        full_entropy_array = np.concatenate((full_entropy_array, entropy))

#save the array in the npz format
np.savez("./entropy_array.npz", full_entropy_array)

/home/riccardo/uni_projects/complex_networks/Abstract_network/extra/components_selection.py:32: RuntimeWarning: divide by zero encountered in log
  embedding_matrix_normalized = np.where(np.isclose(embedding_matrix_normalized, 0.), 0., (embedding_matrix_normalized * np.log(embedding_matrix_normalized)) * (-1))
/home/riccardo/uni_projects/complex_networks/Abstract_network/extra/components_selection.py:32: RuntimeWarning: invalid value encountered in multiply
  embedding_matrix_normalized = np.where(np.isclose(embedding_matrix_normalized, 0.), 0., (embedding_matrix_normalized * np.log(embedding_matrix_normalized)) * (-1))


Now load the array with entropies, and filter the components of the tf-idf embeddings keeping only those with entropy greater than 1.

In [ ]:
entropy_array = np.load("./entropy_array.npz")['arr_0']
filtered_embeddings = tf_idf_embeddings[:, np.where(entropy_array > 1)[0]]

array([4.01704797, 1.7031299 , 1.02075931, ..., 0.69064161, 1.74141953,
       0.692523  ], shape=(55750,))

## Build network and split into communities

Now we can build the distance matrix, the adjacency matrix, the relative network using the same functions we used for the network with all the components.